In [ ]:
import os
import sys
import shutil
import subprocess
import time
from datetime import datetime
from google.colab import files
from IPython.display import display, Markdown

import sys
'''
許多因為碎片化導致的隨機 OOM（例如：訓練到一半突然崩潰）可以得到有效緩解。

這行程式碼必須放在 import torch 之前，或者在程式剛啟動時最先執行，
否則 PyTorch 一旦初始化完成，這個環境變數設定就不會生效。
'''
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
import importlib.metadata as md

In [ ]:
# 安裝編譯 ffmpeg 所需的相依套件
!sudo apt update
!sudo apt install -y build-essential pkg-config yasm nasm curl xz-utils libmp3lame-dev

In [ ]:
# 將 bash 腳本寫入檔案
script = r'''
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive

cd /content
rm -rf ffmpeg-5.1.9 ffmpeg-5.1.9.tar.xz ffmpeg-5.1.9-build

curl -L --progress-bar -o ffmpeg-5.1.9.tar.xz \
  https://ffmpeg.org/releases/ffmpeg-5.1.9.tar.xz

tar -xf ffmpeg-5.1.9.tar.xz
cd ffmpeg-5.1.9

./configure \
  --prefix=/content/ffmpeg-5.1.9-build \
  --enable-shared \
  --disable-static \
  --disable-doc \
  --disable-debug \
  --enable-gpl \
  --enable-libmp3lame

make -j"$(nproc)"
make install
'''

with open("/content/build_ffmpeg5_mp3.sh", "w") as f:
    f.write(script)

!chmod +x /content/build_ffmpeg5_mp3.sh

In [ ]:
# 執行 bash 腳本，並即時顯示輸出
process = subprocess.Popen(
    "stdbuf -oL -eL bash /content/build_ffmpeg5_mp3.sh",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
print("return code:", return_code)

In [ ]:
# 針對 ffmpeg 5.1.9 的動態連結庫，將其路徑加入到系統的動態連結庫搜尋路徑中
!echo /content/ffmpeg-5.1.9-build/lib | sudo tee /etc/ld.so.conf.d/ffmpeg5.conf
!sudo ldconfig

In [ ]:
# 檢查 ffmpeg 版本
!/content/ffmpeg-5.1.9-build/bin/ffmpeg -version | head -n 1

In [ ]:
# 將剛編譯的 ffmpeg 加入 PATH，並將其動態連結庫加入 LD_LIBRARY_PATH，確保系統能找到正確的 ffmpeg 和相關庫。
os.environ["PATH"] = "/content/ffmpeg-5.1.9-build/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = (
    "/content/ffmpeg-5.1.9-build/lib:"
    "/usr/local/lib/python3.12/dist-packages/torch/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print(shutil.which("ffmpeg"))

In [ ]:
# 從 GitHub clone heartlib 倉庫，如果已經存在則跳過。
if not os.path.exists("/content/heartlib"):
    !git clone https://github.com/HeartMuLa/heartlib.git /content/heartlib -q

In [ ]:
# 切換到 heartlib 目錄並安裝相關套件
%cd /content/heartlib
!pip install -q -e .

In [ ]:
# 安裝 torchcodec 套件 (版本 0.10.0，對應 CUDA 12.8 和 PyTorch 2.10.0)
# !pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install torchcodec==0.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
print("python:", sys.version)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)

'''
一般來說會輸出這個訊息:
python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.10.0+cu128
torch cuda: 12.8
torchcodec: 0.10.0+cu128
'''

try:
    print("torchcodec:", md.version("torchcodec"))
except Exception as e:
    print("torchcodec not found:", e)

In [ ]:
# 切換到 heartlib 目錄
%cd /content/heartlib

# 建立 ckpt 目錄，如果已經存在則不會報錯
os.makedirs('./ckpt', exist_ok=True)

# 從 Hugging Face 下載模型檔案，如果已經存在則跳過下載
if not os.path.exists('./ckpt/HeartMuLa-oss-3B'):
    !huggingface-cli download --local-dir './ckpt/HeartMuLa-oss-3B' 'benjiaiplayground/HeartMuLa-oss-3B-bf16' --quiet

    # 可以選擇量化模型（int4）改成下載 PavonicAI/HeartMuLa-3B-4bit
    # !huggingface-cli download --local-dir './ckpt/HeartMuLa-oss-3B' 'PavonicAI/HeartMuLa-3B-4bit' --quiet

if not os.path.exists('./ckpt/HeartCodec-oss'):
    !huggingface-cli download --local-dir './ckpt/HeartCodec-oss' 'benjiaiplayground/HeartCodec-oss-bf16' --quiet

if not os.path.exists('./ckpt/HeartMuLaGen'):
    !huggingface-cli download --local-dir './ckpt' 'HeartMuLa/HeartMuLaGen' --quiet

# 移動模型檔案到指定位置
codec_target_file = './ckpt/HeartCodec-oss/model.safetensors'
codec_source_file = './ckpt/HeartCodec-oss/HeartCodec-oss-bf16.safetensors'
if not os.path.exists(codec_target_file):
     shutil.move(codec_source_file, codec_target_file)

# 列出 ckpt 目錄下的檔案以確認下載和移動是否成功
!ls -la ./ckpt/

## 參考資訊
- HeartMuLa 主頁面：[https://heartmula.github.io/](https://heartmula.github.io/)
- HeartMuLa 風格標籤：[https://heart-mula.com/tw/tags](https://heart-mula.com/tw/tags)

### 常用而且相對穩定的有：
- [Intro]        前奏，通常沒有歌詞或只有短句
- [Verse]        主歌，敘事、鋪陳
- [Verse 1]      第一段主歌
- [Verse 2]      第二段主歌
- [Pre-Chorus]   副歌前段，情緒推高
- [Chorus]       副歌，主題與 hook
- [Post-Chorus]  副歌後延伸，常用重複旋律或關鍵詞
- [Bridge]       橋段，轉折、換情緒、換和弦
- [Interlude]    間奏，段落之間的音樂過場
- [Instrumental] 純樂器段
- [Solo]         樂器 solo，例如 guitar solo
- [Breakdown]    情緒突然變稀疏或降低編制
- [Build]        逐漸堆疊，準備進副歌或 drop
- [Drop]         電子/流行常用，節奏或主旋律爆發
- [Outro]        尾奏、收尾
- [End]          明確要求結束


### 如果是人聲表現，也可以試：
- [Spoken]       口白
- [Rap]          饒舌
- [Ad-lib]       即興尾音、襯詞
- [Backing Vocals] 和聲
- [Duet]         雙人對唱

### 但要注意：越基本的標籤越穩定
- [Verse]、[Chorus]、[Bridge]、[Outro] 通常比很細的 [Build]、[Drop]、[Ad-lib] 更可靠。

### 補充
- 歌詞的文字長度（包括風格標籤），決定運算的時候佔用多少 GPU memory。
- 播放長度決定運算所花費的時間。

---

## 執行指令的版本

In [ ]:
# 切換到 /content/heartlib
%cd /content/heartlib

# 定義歌詞
lyrics = """\
[Intro]

[Verse]
妳挽著他走向圓滿
我把舊夢藏進襯衫
人群笑得那麼燦爛
我卻像丟了答案

[Chorus]
我站在妳幸福的對岸
把眼淚說成祝妳美滿
妳成了別人的新娘
也成了最大的遺憾

[Outro]
酒醒以後天色微藍
街燈陪我走得很慢
妳的名字還在心上
像一枚戒指沒有歸還"""

# tags 可以用來引導模型生成特定風格或元素的音樂
tags = "pop,sad,ballad,baritone"

# 設定生成參數
duration = 90    # 生成音樂的長度（秒）
temperature = 1.0 # 生成的隨機程度，數值越大越隨機
topk = 50         # 生成時考慮的詞彙數量，數值越大越多樣化
cfg_scale = 1.5   # classifier-free guidance 的強度，數值越大越忠於條件（lyrics 和 tags），但可能犧牲多樣性

# 記錄開始時間，生成檔案名稱，並建立 assets 目錄
start_time = time.time()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"./assets/music_{timestamp}.mp3"
os.makedirs("./assets", exist_ok=True)

# 將 lyrics 和 tags 寫入臨時檔案
lyrics_file = f"./assets/lyrics_{timestamp}.txt"
tags_file = f"./assets/tags_{timestamp}.txt"
with open(lyrics_file, "w", encoding="utf-8") as f:
    f.write(lyrics)
with open(tags_file, "w", encoding="utf-8") as f:
    f.write(tags)

# 建立命令列參數，呼叫 run_music_generation.py 生成音樂
cmd = [
    "python", "./examples/run_music_generation.py",
    "--model_path=./ckpt",
    "--version=3B",
    "--lazy_load=true",
    f"--lyrics={lyrics_file}",
    f"--tags={tags_file}",
    f"--max_audio_length_ms={int(duration * 1000)}",
    f"--temperature={temperature}",
    f"--topk={topk}",
    f"--cfg_scale={cfg_scale}",
    f"--save_path={output_path}"
]

# 執行命令並捕捉輸出
result = subprocess.run(cmd, capture_output=True, text=True)

# 清理臨時檔案
try:
    os.remove(lyrics_file)
    os.remove(tags_file)
except:
    pass

# 檢查生成結果，若成功則顯示下載連結，否則顯示錯誤訊息
if result.returncode == 0 and os.path.exists(output_path):
    elapsed = time.time() - start_time
    display(Markdown(f"✅ 生成完成：`{output_path}`  \n耗時：{elapsed/60:.1f} 分鐘"))
    files.download(output_path)
else:
    error_msg = result.stderr if result.stderr else result.stdout
    print("❌ 生成失敗")
    print(error_msg[-2000:])


---

## 建立 Web API 讓 client 端取得生成結果

In [ ]:
# 安裝 Flask 和 pyngrok 套件，用於建立 Web API 服務
!pip -q install flask pyngrok

In [ ]:
# 檢查 GPU 狀態，確認 CUDA 是否可用
!nvidia-smi

In [ ]:
%%writefile /content/heartlib/api.py
import os, uuid, threading, torch
from flask import Flask, request, send_file, jsonify
from werkzeug.utils import secure_filename
from heartlib import HeartMuLaGenPipeline

MODEL_PATH = "/content/heartlib/ckpt"
OUT_DIR = "/content/heartlib/outputs"
PORT = 5000

os.makedirs(OUT_DIR, exist_ok=True)

app = Flask(__name__)
jobs = {}

pipe = HeartMuLaGenPipeline.from_pretrained(
    MODEL_PATH,
    device={"mula": torch.device("cuda"), "codec": torch.device("cuda")},
    dtype={"mula": torch.bfloat16, "codec": torch.float32},
    version="3B",
    lazy_load=True,
)

def run_job(job_id, lyrics_path, tags_path, out_path, params):
    try:
        jobs[job_id]["status"] = "running"

        with torch.no_grad():
            pipe(
                {"lyrics": lyrics_path, "tags": tags_path},
                save_path=out_path,
                max_audio_length_ms=params["max_audio_length_ms"],
                topk=params["topk"],
                temperature=params["temperature"],
                cfg_scale=params["cfg_scale"],
            )

        jobs[job_id]["status"] = "done"

    except Exception as e:
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = str(e)

@app.get("/health")
def health():
    return jsonify({"status": "ok"})

@app.post("/generate")
def generate():
    job_id = uuid.uuid4().hex
    job_dir = os.path.join(OUT_DIR, job_id)
    os.makedirs(job_dir, exist_ok=True)

    lyrics_path = os.path.join(job_dir, "lyrics.txt")
    tags_path = os.path.join(job_dir, "tags.txt")
    out_path = os.path.join(job_dir, "song.mp3")

    request.files["lyrics_file"].save(lyrics_path)
    request.files["tags_file"].save(tags_path)

    filename = request.form.get("filename", "song.mp3")
    filename = secure_filename(filename)
    if not filename.lower().endswith(".mp3"):
        filename += ".mp3"

    params = {
        "max_audio_length_ms": int(request.form.get("max_audio_length_ms", 60000)),
        "topk": int(request.form.get("topk", 50)),
        "temperature": float(request.form.get("temperature", 1.0)),
        "cfg_scale": float(request.form.get("cfg_scale", 1.5)),
    }

    jobs[job_id] = {
        "status": "queued",
        "filename": filename,
        "out_path": out_path,
    }

    threading.Thread(
        target=run_job,
        args=(job_id, lyrics_path, tags_path, out_path, params),
        daemon=True,
    ).start()

    return jsonify({
        "job_id": job_id,
        "status_url": f"/status/{job_id}",
        "download_url": f"/download/{job_id}",
    })

@app.get("/status/<job_id>")
def status(job_id):
    if job_id not in jobs:
        return jsonify({"error": "job not found"}), 404
    return jsonify(jobs[job_id])

@app.get("/download/<job_id>")
def download(job_id):
    if job_id not in jobs:
        return jsonify({"error": "job not found"}), 404

    job = jobs[job_id]

    if job["status"] != "done":
        return jsonify({"status": job["status"]}), 202

    return send_file(
        job["out_path"],
        mimetype="audio/mpeg",
        as_attachment=True,
        download_name=job["filename"],
    )

app.run(host="0.0.0.0", port=PORT, debug=False, threaded=True)

In [ ]:
# 開啟 ngrok，將本地的 Flask 服務暴露到公網，方便外部訪問
from pyngrok import ngrok
from google.colab import userdata
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if not token:
    token = getpass("NGROK_AUTHTOKEN: ")

ngrok.set_auth_token(token)
ngrok.kill()

url = ngrok.connect(5000, "http").public_url

print("URL:", url)
print("Health:", url + "/health")
print("Generate:", url + "/generate")

In [ ]:
# 啟動 Flask Web API 服務
!python api.py